# M1 — Computer vision

**NTH bootcamp · Module 1**

Before a cortical visual prosthesis can stimulate the brain, it needs to decide *what to light up*. This notebook walks through the front-end of that pipeline:

1. Image I/O and preprocessing
2. Classical edge detection (Sobel, Canny, threshold)
3. Deep segmentation with YOLO
4. Combining edges with segmentation
5. Extensions and a final challenge

The visual companion to this notebook is [`M1-computer-vision.html`](https://github.com/NeuroTechHub/AIMD_bootcamp/blob/main/modules/M1-computer-vision.html) — no programming required, useful for warming up.

Exercises are tagged **`[easy]`**, **`[intermediate]`**, or **`[challenge]`**. The challenges are genuinely hard; do not feel obliged to finish them in the guided hour.

## Setup

Required packages:

```bash
pip install numpy opencv-python matplotlib ultralytics
```

The cell below installs them, imports everything, downloads `bus.jpg` if it isn't already in `assets/`, and defines a small `show()` helper used throughout. Run it once.

> **Colab vs. local:** the install cell auto-detects the environment. On a local Anaconda base it adds `--user`; on Colab it installs without `--user` (the `--user` flag installs to a path that isn't importable on Colab). If `import ultralytics` still fails right after install, use **Runtime → Restart runtime** and re-run.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../build/assets').resolve()))
from notebook_setup import ensure
ensure(['numpy', 'opencv-contrib-python', 'matplotlib', 'ultralytics'])

import importlib
importlib.invalidate_caches()
import ultralytics
print('ultralytics', ultralytics.__version__)


In [ ]:
import os, sys, urllib.request, shutil
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

ASSETS = Path('assets')
ASSETS.mkdir(exist_ok=True)
BUS = ASSETS / 'bus.jpg'
if not BUS.exists():
    # bus.jpg ships inside the ultralytics package, so use that copy first (no network,
    # avoids the flaky ultralytics.com redirect). Fall back to the URL only if needed.
    try:
        from ultralytics.utils import ASSETS as _ULTRA_ASSETS
        shutil.copy(_ULTRA_ASSETS / 'bus.jpg', BUS)
        print(f'copied {BUS} from ultralytics assets')
    except Exception:
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', BUS)
        print(f'downloaded {BUS}')

# Named constants. Same values the HTML page exposes as sliders; collected
# here so the next cells read as "what we're doing", not opaque literals.
RESIZE_SIDE = 224          # ImageNet input convention.
BLUR_KSIZE  = (5, 5)       # short-range smoothing before edges (Canny 1986).
BLUR_SIGMA  = 1.4          # ~1 px std-dev; smaller misses noise, larger blurs detail.
CANNY_LO    = 50           # weak-edge hysteresis threshold (Canny 1986: ratio ~1:3).
CANNY_HI    = 150          # strong-edge hysteresis threshold.
THRESH      = 128          # midpoint binary threshold (uint8 dynamic range).

# Loud fallback banner shared across modules. Used so a model swap (YOLO
# below, MiDaS in M4) prints a high-contrast notice instead of slipping by.
sys.path.insert(0, str(Path('../../build/assets').resolve()))
try:
    from fallback_banner import loud_fallback
except ImportError:
    def loud_fallback(*, real, stand_in, reason='', losing=''):
        print(f'[FALLBACK] {real} -> {stand_in}: {reason}')

def show(*imgs, titles=None, cmap='gray', figsize=None, cols=None):
    """Plot one or more images side-by-side. BGR images are auto-converted to RGB."""
    if len(imgs) == 1 and isinstance(imgs[0], (list, tuple)):
        imgs = imgs[0]
    n = len(imgs)
    cols = cols or n
    rows = int(np.ceil(n / cols))
    figsize = figsize or (4*cols, 4*rows)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis('off')
        if i >= n: continue
        im = imgs[i]
        if im.ndim == 3 and im.shape[2] == 3:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(im, cmap=cmap, vmin=0, vmax=im.max() if im.dtype != np.uint8 else 255)
        if titles and i < len(titles):
            ax.set_title(titles[i], fontsize=10)
    plt.tight_layout(); plt.show()

print('versions:', 'numpy', np.__version__, '· cv2', cv2.__version__)

## 1 · Image basics

An image is a NumPy array. Conventions to keep in mind:

| Library | Channel order | dtype | Shape |
|---|---|---|---|
| `cv2.imread` | **BGR** | uint8 | `(H, W, 3)` |
| `PIL.Image.open` / `matplotlib.imread` | RGB | uint8 or float | `(H, W, 3)` |
| Grayscale | — | uint8 | `(H, W)` |

In [ ]:
bgr = cv2.imread(str(BUS))
print('shape', bgr.shape, '· dtype', bgr.dtype, '· min/max', bgr.min(), bgr.max())
show(bgr, titles=['bus.jpg'])

### Exercise 1.1 — resize, grayscale, plot `[easy]`

Take `bgr`, resize it to **224 × 224** with `cv2.resize`, convert to grayscale with `cv2.cvtColor` and `cv2.COLOR_BGR2GRAY`, and plot both versions side-by-side. Print the shape and dtype of the result.

> Hint: `cv2.resize(img, (W, H))` — note that the size is `(width, height)`, not `(height, width)`.

In [ ]:
# your code here


### Exercise 1.2 — convert between pixel formats `[easy]`

OpenCV stores pixels as integers from 0 to 255 (`uint8`), but most ML models expect floating-point numbers between 0 and 1. You'll need to switch between the two.

Write two short functions:

* `as_float(img)` — takes a `uint8` image and returns a `float32` image with values in `[0, 1]`.
* `as_uint8(img)` — takes a `float32` image with values in `[0, 1]` and returns a `uint8` image with values in `[0, 255]`.

Then apply both in sequence to `bgr` (convert to float, then back to `uint8`) and check that the result is exactly the same as the original. `np.array_equal(original, restored)` should print `True`.

In [ ]:
# your code here


## 2 · Edge detection

Three classical methods, in order of increasing sophistication:

| Method | Idea | When to use |
|---|---|---|
| **Threshold** | Brightness < level → 0, else 255 | Foreground is uniformly darker or brighter than background. Documents, blob-tracking. |
| **Sobel** | Approximate the brightness gradient with a 3×3 kernel; magnitude = edge strength | Quick first look. Gives continuous values, not binary. Sensitive to noise. |
| **Canny** | Blur → Sobel → non-max suppression → hysteresis | Clean, thin, connected edges. The default for most pipelines. Needs two thresholds. |

Let's run all three on the bus image.

In [ ]:
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, BLUR_KSIZE, BLUR_SIGMA)

# Threshold
_, th_bin = cv2.threshold(blur, THRESH, 255, cv2.THRESH_BINARY)
_, th_otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Sobel
sx = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
sy = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)
smag = np.sqrt(sx**2 + sy**2)
smag = np.clip(smag / smag.max() * 255, 0, 255).astype(np.uint8)

# Canny
canny = cv2.Canny(blur, CANNY_LO, CANNY_HI)

show(gray, th_bin, th_otsu, smag, canny,
     titles=['grayscale', f'threshold @{THRESH}', 'Otsu', 'Sobel magnitude',
             f'Canny ({CANNY_LO},{CANNY_HI})'],
     cols=5, figsize=(20, 5))

Notice:

* Plain threshold catches the bus body but loses the dark people against the dark bus — a flat brightness cut can't separate them.
* Sobel responds to every gradient, including pavement texture.
* Canny suppresses everything that isn't a *local* maximum of gradient *and* connected to a strong edge — giving clean outlines.

### Exercise 2.1 — sensitivity to thresholds `[easy]`

Run `cv2.Canny` with three different threshold pairs:

* `(20, 60)` — very permissive
* `(50, 150)` — the default we just used
* `(100, 300)` — strict

Plot the three results side-by-side and write one sentence about how the output changes.

In [ ]:
# your code here


### Exercise 2.2 — Sobel from scratch `[intermediate]`

The Sobel edge detector is just two small filters slid across the image. One filter (`Gx`) measures how much brightness changes from left to right; the other (`Gy`) measures how much it changes from top to bottom. The "edge strength" at a pixel tells you how big those two changes are *together*.

The filters look like this:

```
Gx = [[-1, 0, +1],          Gy = [[-1, -2, -1],
      [-2, 0, +2],                [ 0,  0,  0],
      [-1, 0, +1]]                [+1, +2, +1]]
```

Your task:

1. Write each filter as a NumPy array (use `dtype=np.float32`).
2. Apply each filter to `blur` with `cv2.filter2D`. Pass `ddepth=cv2.CV_32F` so negative values aren't lost.
3. Combine the two results into a single edge-strength map. *Hint:* think of `gx` and `gy` as the two sides of a right triangle — the edge strength is the length of the hypotenuse.
4. Check that your result matches `cv2.Sobel` by printing the maximum absolute difference; it should be ~0.

In [ ]:
# Exercise 2.2 — Sobel from scratch
# 1) Write the two 3x3 kernels as float32 NumPy arrays
Kx = ...   # your code here
Ky = ...   # your code here

# 2) Apply each kernel to `blur` with cv2.filter2D
#    Hint: cv2.filter2D(blur, cv2.CV_32F, kernel)
gx = ...   # your code here
gy = ...   # your code here

# 3) Combine gx and gy into a single edge-strength map
mag = ...  # your code here

# 4) Compare to cv2.Sobel — max difference should be ~0
gx_cv = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
gy_cv = cv2.Sobel(blur, cv2.CV_32F, 0, 1, ksize=3)
mag_cv = np.sqrt(gx_cv**2 + gy_cv**2)
# print('max difference:', np.abs(mag - mag_cv).max())


## 3 · YOLO segmentation

Classical edges are blind to *what* an object is. A pretrained segmentation network gives us per-pixel object masks plus a class label. We use **YOLOv8n-seg** (the smallest segmentation variant — ~6 MB, runs on CPU in under a second).

The cell below runs the model. If `ultralytics` isn't installed, or you have no internet, it falls back to the precomputed masks shipped in `assets/bus_masks.npz` and the rest of the notebook continues to work.

In [ ]:
yolo_ok = False
try:
    from ultralytics import YOLO
    model = YOLO('yolov8n-seg.pt')  # downloads on first run
    result = model.predict(str(BUS), verbose=False)[0]
    yolo_ok = True
    print(f'YOLO ran. {len(result.boxes)} detections.')
except Exception as e:
    loud_fallback(
        real='YOLO v8n-seg object segmentation',
        stand_in='precomputed masks in assets/bus_masks.npz',
        reason=str(e),
        losing='live class labels and per-pixel masks; only the cached bus scene works',
    )
    cache = np.load(ASSETS / 'bus_masks.npz', allow_pickle=True)
    print('available arrays:', list(cache.files))

In [ ]:
# unified accessor: works whether YOLO ran or we loaded the cache
import json

def get_detections():
    """Return (masks, classes, confs, class_names).
    masks: (N, H, W) uint8 binary; classes: (N,) int; confs: (N,) float; class_names: dict[int,str]"""
    if yolo_ok:
        H, W = bgr.shape[:2]
        raw = result.masks.data.cpu().numpy()
        masks = np.stack([
            (cv2.resize(m, (W, H), interpolation=cv2.INTER_LINEAR) > 0.5).astype(np.uint8)
            for m in raw
        ])
        classes = result.boxes.cls.cpu().numpy().astype(int)
        confs = result.boxes.conf.cpu().numpy()
        names = result.names
    else:
        # Offline path: per-detection PNG masks listed in bus_detections.json
        cache = np.load(ASSETS / 'bus_masks.npz', allow_pickle=True)
        names = {i: n for i, n in enumerate(cache['class_names'])}
        name_to_id = {n: i for i, n in names.items()}

        with open(ASSETS / 'bus_detections.json') as f:
            det_json = json.load(f)

        masks_list, classes_list, confs_list = [], [], []
        for d in det_json['detections']:
            m = cv2.imread(str(ASSETS / d['mask']), cv2.IMREAD_GRAYSCALE)
            masks_list.append((m > 127).astype(np.uint8))
            classes_list.append(name_to_id[d['class']])
            confs_list.append(d['conf'])

        masks = np.stack(masks_list)
        classes = np.array(classes_list, dtype=int)
        confs = np.array(confs_list, dtype=float)
    return masks, classes, confs, names

masks, classes, confs, names = get_detections()
for m, c, p in zip(masks, classes, confs):
    print(f'  {names[int(c)]:12s}  conf={p:.2f}  mask area={int(m.sum())}px')

In [ ]:
# overlay all masks on the original image, one colour per detection
overlay = bgr.copy()
rng = np.random.default_rng(0)
for m in masks:
    colour = rng.integers(50, 255, size=3).tolist()
    overlay[m.astype(bool)] = (0.55 * np.array(colour) + 0.45 * overlay[m.astype(bool)]).astype(np.uint8)
show(bgr, overlay, titles=['original', 'YOLO segmentation overlay'])

### Exercise 3.1 — class roster `[easy]`

Print a table of every detection: index, class name, confidence, pixel area. Sort by confidence descending. One line per detection. Use just NumPy and an f-string — no pandas needed.

In [ ]:
# Exercise 3.1 — class roster sorted by confidence
#
# You already have these arrays from the cell above:
#   classes  : shape (N,)        int    -- class id of each detection (e.g. 0)
#   confs    : shape (N,)        float  -- confidence score, in [0, 1]
#   masks    : shape (N, H, W)   uint8  -- binary mask per detection
#                                          (use masks[i].sum() to get pixel area)
#   names    : dict[int, str]            -- map class id -> name, e.g. names[0] == 'person'
#
# Useful numpy bits:
#   - np.argsort(arr)    -> indices that would sort `arr` ascending
#   - np.argsort(-arr)   -> ... descending (trick: negate first)
#   - len(classes)       -> N, the number of detections
#
# Then loop over the sorted indices and print one line per detection.

# your code here

### Exercise 3.2 — class-filtered union mask `[intermediate]`

Build a single binary mask `people_mask` that is `1` wherever **any** person is detected and `0` everywhere else. Then build `vehicles_mask` for the `bus` class (extend this later to `car`, `truck` if you swap images). Plot the three on top of the original.

> Hint: `cls_idx = [i for i, c in enumerate(classes) if names[int(c)] == 'person']` and then `np.any(masks[cls_idx], axis=0)`.

In [ ]:
# Exercise 3.2 — class-filtered union masks
#
# Same arrays as 3.1 are available: classes, confs, masks, names.
#
# Steps:
#   1) Find the indices i where the class name is 'person':
#        cls_idx = [i for i, c in enumerate(classes) if names[int(c)] == 'person']
#   2) Combine those masks into one binary mask of shape (H, W):
#        people_mask = np.any(masks[cls_idx], axis=0)
#   3) Repeat for 'bus' -> vehicles_mask.
#   4) Plot the original image plus an overlay for each mask
#      (use show(...) or build a tinted overlay with mask indexing).

# your code here

## 4 · Combining edges and segmentation

You now have two views of the scene: a "what's where" mask from YOLO and a fine-geometry edge map from Canny. The exercises below combine them in different ways.

First, a quick recap — the binary union of every YOLO mask, and the Canny edges on the original image.

In [ ]:
edge_map = cv2.Canny(blur, CANNY_LO, CANNY_HI)
object_map = np.any(masks.astype(bool), axis=0).astype(np.uint8) * 255

show(edge_map, object_map, titles=['edges (Canny)', 'object union (YOLO)'])

Side by side, the two views encode different priorities — the edge map preserves fine geometry (vehicle outline, road), the object map preserves *what* matters but discards detail inside it. Neither is strictly better.

### Exercise 4.1 — show each binary mask `[easy]`

YOLO gave you a stack of binary masks — one per detection. Display them so you can see what was found.

For each detection `i`, plot `masks[i]` (a `(H, W)` array of 0s and 1s) and use the class name `names[int(classes[i])]` as the title.

Tip: `show(*list_of_masks, titles=list_of_strings, cols=3)` lays them out in a grid.

In [ ]:
# Exercise 4.1 — show each binary mask
# You have: masks (N, H, W) uint8 (values 0 or 1), classes (N,) int, names dict[int,str].
# Build a list of titles (one class name per detection).
# Multiply masks by 255 before plotting — show() scales uint8 as 0..255,
# so 0/1 masks would look almost black otherwise.

# your code here

### Exercise 4.2 — weighted activation by class `[intermediate]`

Not every object matters equally to a prosthesis user — a person probably matters more than a passing skateboard. We can encode that by giving each class its own weight in `[0, 1]` and rendering a map whose brightness at every pixel reflects the weight of whichever object covers it.

1. Define a `weights` dict mapping class names to floats, e.g. `{'person': 1.0, 'bus': 0.5, 'skateboard': 0.2}`. Any class not in the dict counts as 0.
2. Build a `(H, W)` float map by "painting" each detection's weight onto a blank canvas wherever its mask is 1. Where masks overlap, keep the **larger** weight.
3. Display the result.

Visually: the people should be the brightest, the bus dimmer, the skateboard dimmer still, and the background black.

In [ ]:
# Exercise 4.2 — weighted activation by class
weights = {'person': 1.0, 'bus': 0.5, 'skateboard': 0.2}

# You have:
#   masks   (N, H, W) uint8 binary
#   classes (N,) int     -- detection class ids
#   names   dict[int,str]
#   bgr.shape[:2] -> (H, W)
#
# Steps:
#   1) Make a float32 canvas of zeros, shape (H, W).
#   2) For each detection i:
#        - look up its class name -> weight (default 0 if not in `weights`)
#        - update the canvas with the elementwise max of canvas and (weight * masks[i])
#          Hint: np.maximum(canvas, weight * masks[i].astype(np.float32))
#   3) Display the weighted canvas. Multiply by 255 first since the values live in [0, 1]
#      and show() scales uint8 as 0..255.

# your code here

### Exercise 4.3 — salient edges `[intermediate]`

Combine edges and segmentation into one activation map: **only edges that fall inside a YOLO mask survive**. This is one way to get the best of both worlds — fine detail (the outline of the bus's windows, a person's silhouette) but no clutter (no tree leaves, no pavement texture).

1. Build `obj_union` — the binary union of all YOLO masks.
2. Compute `canny_edges` of the original image.
3. `salient = canny_edges * obj_union` (mask the edges).
4. Pool to the electrode grid.
5. Plot the salient activation alongside the plain edge activation. Note the difference.

Bonus: dilate the masks slightly with `cv2.dilate` first, so edges *on the boundary* of objects (which often fall just outside the segmentation) are not lost.

In [ ]:
# Exercise 4.3 — salient edges (edges restricted to YOLO masks)
# You have: masks, blur.
#
# Steps:
#   1) obj_union = np.any(masks.astype(bool), axis=0).astype(np.uint8)
#   2) Optional: dilate obj_union with cv2.dilate so boundary edges survive
#   3) canny_edges = cv2.Canny(blur, CANNY_LO, CANNY_HI)
#   4) salient = canny_edges * obj_union
#   5) show() the salient edges next to the plain canny edges.

# your code here

## 5 · Extensions

Optional. Pick one if you finish early — these are the seeds for the "vibe coding" hour later in the day.

### A — Webcam
Below is a ready-to-run loop that opens your webcam and shows the live feed next to a Canny edge map. Flip `RUN_WEBCAM` to `True`, run the cell, and press `q` in the camera window to quit.

Once it works, **add YOLO**: re-use what you wrote in §3 to detect objects per frame and overlay the masks. Hints:
- YOLO is slow on CPU. Don't run it every frame — run it once every 5–10 frames and re-use the last `masks` / `classes` in between.
- `model.predict(frame, verbose=False)[0]` accepts a `(H, W, 3)` BGR numpy array directly — no resizing or colour conversion needed first.
- Downsize the frame to ~320×320 before passing it to YOLO to keep the loop responsive.


In [ ]:
# A — Webcam: live feed + Canny edges.
# Flip the flag to True, run the cell, press 'q' in the window to quit.
RUN_WEBCAM = False

if RUN_WEBCAM:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Couldn't open the webcam.")

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(cv2.GaussianBlur(gray, BLUR_KSIZE, BLUR_SIGMA), CANNY_LO, CANNY_HI)
        edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

        cv2.imshow('webcam (left) | canny (right) — press q to quit',
                   np.hstack([frame, edges_bgr]))
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

### B — Open-vocabulary detection
`yolov8n-seg` is locked to the 80 COCO classes. **Open-vocabulary** models accept *text* prompts and find whatever you describe — "a person carrying a backpack", "a fire hydrant", "a striped umbrella" — without retraining.

**Get the model.** Two good options, both delivered via the `ultralytics` package you already installed:

| Model | Weights | Notes |
|---|---|---|
| [YOLO-World](https://docs.ultralytics.com/models/yolo-world/) | `yolov8s-world.pt` (~50 MB) | Drop-in replacement; downloaded on first use. |
| [YOLOE](https://docs.ultralytics.com/models/yoloe/) | `yoloe-11s-seg.pt` (~60 MB) | Open-vocab **with segmentation masks**. |

Both auto-download into the current directory on first call — no manual download needed, as long as you have internet.

The pattern is: load the model, tell it what classes you care about with `set_classes(...)`, then predict as usual. A skeleton:

```python
from ultralytics import YOLO

ov_model = YOLO('yolov8s-world.pt')      # downloads on first run

# Tell it what to look for. These don't have to be COCO classes.
ov_model.set_classes(['stop sign', 'crosswalk', 'pedestrian crossing sign'])

result = ov_model.predict(str(BUS), verbose=False)[0]
# result.boxes  -> same shape as before
# result.masks  -> only if you used a segmentation variant (e.g. yoloe-11s-seg.pt)
```

Try a few class lists. What can your prosthesis stimulate now that it couldn't with COCO?


In [ ]:
# B — Open-vocabulary detection: skeleton.
# Set TRY_OPENVOCAB = True to download the model (~50 MB) and run it.
TRY_OPENVOCAB = False

if TRY_OPENVOCAB:
    from ultralytics import YOLO

    ov_model = YOLO('yolov8s-world.pt')                       # downloads on first run
    ov_model.set_classes(['stop sign', 'crosswalk', 'umbrella'])

    ov_result = ov_model.predict(str(BUS), verbose=False)[0]
    print(f'{len(ov_result.boxes)} detections for the prompt list.')

    # If you want masks too, use a segmentation variant instead:
    #   ov_model = YOLO('yoloe-11s-seg.pt')
    # then ov_result.masks.data is a (N, h, w) tensor in [0, 1].

### C — Final challenge: weighted real-time vision `[challenge]`
Bolt your webcam loop (A) onto the weighted-by-class map from exercise 4.2. Per frame:

1. Run YOLO every few frames; cache the resulting `masks` and `classes`.
2. Build the weighted map from the cached detections using the same `weights` dict from 4.2.
3. (Optional) Intersect it with this frame's Canny edges, à la 4.3 (salient edges).
4. Show the raw frame next to the weighted map in a single window.

Aim for ≥ 10 fps on CPU. Tip: keep YOLO inference at 320×320 or smaller; only the *display* needs to be full-resolution.

Anything you build here is reusable later.


---

**Done.** You've gone from raw pixels to edge maps and per-object segmentation masks, and combined them in a few different ways. Bring whichever output you like best into the next module.

Module lead: Lefteris & Jorge. Edit this notebook directly; commit your additions to the bootcamp repo at the end of the day.

## Notebook self-check

Run this after you have filled the exercise cells. It only checks variables that
exist in your kernel, so a fresh blank notebook prints `nothing to check yet`.
When a check fires, it catches shape/range mistakes before you compare against
the solution notebook.


In [ ]:
checks = []
if 'gray224' in globals():
    assert gray224.shape == (224, 224), f'gray224 shape is {gray224.shape}, expected (224, 224)'
    checks.append('1.1 gray224 shape')
if 'as_float' in globals() and 'as_uint8' in globals():
    f = as_float(np.array([0, 128, 255], dtype=np.uint8))
    assert f.dtype.kind == 'f' and np.all((0 <= f) & (f <= 1)), 'as_float should return float values in [0, 1]'
    assert as_uint8(f).dtype == np.uint8, 'as_uint8 should return uint8'
    checks.append('1.2 dtype conversion')
if 'Kx' in globals() and 'Ky' in globals():
    assert np.asarray(Kx).shape == (3, 3) and np.asarray(Ky).shape == (3, 3), 'Sobel kernels should be 3x3'
    checks.append('2.2 Sobel kernels')
if 'people_mask' in globals():
    assert people_mask.ndim == 2 and people_mask.max() <= 1, 'people_mask should be a 2D binary mask'
    checks.append('3.2 people mask')
if 'activation' in globals():
    assert activation.ndim == 2 and activation.max() <= 255, 'activation should be a 2D image-like map'
    checks.append('4.2 activation map')
print('Self-checks passed:', ', '.join(checks) if checks else 'nothing to check yet')
